In [625]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor, LGBMClassifier
import sys
import re
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
from scipy.stats import spearmanr

In [626]:
df_listings = pd.read_csv("../data/Listing/listed_transactions.csv")
df_listings.head()

C:\Users\mayab\AppData\Local\Temp\ipykernel_13868\3410094314.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listings = pd.read_csv("../data/Listing/listed_transactions.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,Year
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,NaN,2.0,Covina Valley Unified,91722,NaN,0.0,NaN,NaN,1045 N Azusa 61,2024
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,NaN,NaN,NaN,92025,NaN,0.0,NaN,NaN,NaN,2024
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.0,177861.0,NaN,2220 Avenue Of The Stars 2704,2024
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.0,5300.0,NaN,16 Palisades,2024
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,2024


##### Number of Rows and Columns

In [627]:
df_listings.shape

(892774, 85)

The listing transactions file has 853754 rows and 87 columns. 

##### Review column data types

In [628]:
df_listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 892774 entries, 0 to 892773
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   OriginalListPrice             889335 non-null  float64
 1   ListingKey                    892774 non-null  int64  
 2   ListAgentEmail                819336 non-null  str    
 3   CloseDate                     260134 non-null  str    
 4   ClosePrice                    235942 non-null  float64
 5   ListAgentFirstName            887511 non-null  str    
 6   ListAgentLastName             892697 non-null  str    
 7   Latitude                      780321 non-null  float64
 8   Longitude                     781047 non-null  float64
 9   UnparsedAddress               890413 non-null  str    
 10  PropertyType                  892774 non-null  str    
 11  LivingArea                    782032 non-null  float64
 12  ListPrice                     890556 non-null  float64


In [629]:
df_listings.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency',
       'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus',
       'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1',
       'ListingId', 'BathroomsTotalInteger', 'Cit

In [630]:
df_listings.isna().sum()

OriginalListPrice                 3439
ListingKey                           0
ListAgentEmail                   73438
CloseDate                       632640
ClosePrice                      656832
                                 ...  
AssociationFee                  288402
LotSizeSquareFeet                82123
MiddleOrJuniorSchoolDistrict    892774
UnparsedAddress.1                 2361
Year                                 0
Length: 85, dtype: int64

#### **Missing Value Analysis**

##### Calculate missing counts and percentages per column

In [631]:
missing_listing_counts = df_listings.isnull().sum()
missing_listing_counts

OriginalListPrice                 3439
ListingKey                           0
ListAgentEmail                   73438
CloseDate                       632640
ClosePrice                      656832
                                 ...  
AssociationFee                  288402
LotSizeSquareFeet                82123
MiddleOrJuniorSchoolDistrict    892774
UnparsedAddress.1                 2361
Year                                 0
Length: 85, dtype: int64

In [632]:
missing_listing_percent = (df_listings.isnull().mean()) * 100
missing_listing_percent

OriginalListPrice                 0.385204
ListingKey                        0.000000
ListAgentEmail                    8.225822
CloseDate                        70.862279
ClosePrice                       73.572035
                                   ...    
AssociationFee                   32.304032
LotSizeSquareFeet                 9.198633
MiddleOrJuniorSchoolDistrict    100.000000
UnparsedAddress.1                 0.264457
Year                              0.000000
Length: 85, dtype: float64

In [633]:
missing_summary = pd.DataFrame({
    "missing_listing_counts": missing_listing_counts,
    "missing_listing_percent": missing_listing_percent
})

In [634]:
missing_listing_summary = missing_summary.sort_values(by="missing_listing_percent", ascending=False)
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
MiddleOrJuniorSchoolDistrict                  892774                    100.0
ElementarySchoolDistrict                      892774                    100.0
CoveredSpaces                                 892774                    100.0
AboveGradeFinishedArea                        892774                    100.0
FireplacesTotal                               892774                    100.0
...                                              ...                      ...
PropertyType.1                                     0                      0.0
ListingKeyNumeric                                  0                      0.0
DaysOnMarket.1                                     0                      0.0
ListingContractDate                                0                      0.0
Year                                               0                      0.0

[85 rows x 2 columns]


In [635]:
missing_listing_summary = missing_listing_summary[missing_listing_summary["missing_listing_counts"] > 0]
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
MiddleOrJuniorSchoolDistrict                  892774               100.000000
ElementarySchoolDistrict                      892774               100.000000
CoveredSpaces                                 892774               100.000000
AboveGradeFinishedArea                        892774               100.000000
FireplacesTotal                               892774               100.000000
...                                              ...                      ...
PostalCode                                       218                 0.024418
ListAgentLastName                                 77                 0.008625
ListAgentLastName.1                               77                 0.008625
StateOrProvince                                   71                 0.007953
CountyOrParish                                     1                 0.000112

[74 rows x 2 columns]


In [636]:
missing_listing_summary["missing_listing_percent"] = missing_listing_summary["missing_listing_percent"].round(2)
print(missing_listing_summary)

                              missing_listing_counts  missing_listing_percent
MiddleOrJuniorSchoolDistrict                  892774                   100.00
ElementarySchoolDistrict                      892774                   100.00
CoveredSpaces                                 892774                   100.00
AboveGradeFinishedArea                        892774                   100.00
FireplacesTotal                               892774                   100.00
...                                              ...                      ...
PostalCode                                       218                     0.02
ListAgentLastName                                 77                     0.01
ListAgentLastName.1                               77                     0.01
StateOrProvince                                   71                     0.01
CountyOrParish                                     1                     0.00

[74 rows x 2 columns]


In [637]:
missing_above_90 = missing_listing_summary[missing_listing_summary['missing_listing_percent'] > 90]
print(missing_above_90)

                              missing_listing_counts  missing_listing_percent
MiddleOrJuniorSchoolDistrict                  892774                   100.00
ElementarySchoolDistrict                      892774                   100.00
CoveredSpaces                                 892774                   100.00
AboveGradeFinishedArea                        892774                   100.00
FireplacesTotal                               892774                   100.00
TaxYear                                       891875                    99.90
BelowGradeFinishedArea                        889157                    99.59
BusinessType                                  886542                    99.30
TaxAnnualAmount                               886427                    99.29
CoBuyerAgentFirstName                         872768                    97.76
BuilderName                                   862241                    96.58
LotSizeDimensions                             841763            

In [638]:
missing_above_70 = missing_listing_summary[missing_listing_summary['missing_listing_percent'] > 70]
print(missing_above_70)

                              missing_listing_counts  missing_listing_percent
MiddleOrJuniorSchoolDistrict                  892774                   100.00
ElementarySchoolDistrict                      892774                   100.00
CoveredSpaces                                 892774                   100.00
AboveGradeFinishedArea                        892774                   100.00
FireplacesTotal                               892774                   100.00
TaxYear                                       891875                    99.90
BelowGradeFinishedArea                        889157                    99.59
BusinessType                                  886542                    99.30
TaxAnnualAmount                               886427                    99.29
CoBuyerAgentFirstName                         872768                    97.76
BuilderName                                   862241                    96.58
LotSizeDimensions                             841763            

In [639]:
missing_above_70.shape

(31, 2)

In [640]:
missing_above_70

,missing_listing_counts,missing_listing_percent
MiddleOrJuniorSchoolDistrict,892774,100.00
ElementarySchoolDistrict,892774,100.00
CoveredSpaces,892774,100.00
AboveGradeFinishedArea,892774,100.00
FireplacesTotal,892774,100.00
TaxYear,891875,99.90
BelowGradeFinishedArea,889157,99.59
BusinessType,886542,99.30
TaxAnnualAmount,886427,99.29
CoBuyerAgentFirstName,872768,97.76


To perserve the variables, it would be sufficient if each variable contain less than 70% of missing values in order to use imputation to replace the missingness in the data. 

So, variables with 70% of missing variables will be removed from the dataset because through imputation with variables with a significant number of missingness, it may contribute to bias towards certain values than others. 

Keep these key variables (from above 70):
- ClosePrice
- CloseDate

In [641]:
df_listings_clean = df_listings.drop(columns=['MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict', 'CoveredSpaces',
                                             'AboveGradeFinishedArea', 'FireplacesTotal', 'TaxYear', 
                                             'BelowGradeFinishedArea', 'BusinessType', 'TaxAnnualAmount',
                                             'CoBuyerAgentFirstName', 'BuilderName', 'LotSizeDimensions',
                                             'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool',
                                             'BuyerAgencyCompensation', 'BuyerAgencyCompensationType', 'BuildingAreaTotal',
                                             'CoListAgentFirstName', 'CoListAgentLastName', 'CoListOfficeName', 'BuyerOfficeAOR',
                                            'BuyerOfficeName.1', 'AssociationFeeFrequency', 'BuyerAgentFirstName',
                                             'BuyerAgentMlsId', 'BuyerAgentLastName', 'CloseDate.1'])

Dropping Variables:
- MiddleOrJuniorSchoolDistrict                  
- ElementarySchoolDistrict
- CoveredSpaces
- AboveGradeFinishedArea
- FireplacesTotal
- TaxYear
- BelowGradeFinishedArea
- BusinessType
- TaxAnnualAmount
- CoBuyerAgentFirstName
- BuilderName
- LotSizeDimensions
- ElementarySchool
- MiddleOrJuniorSchool

In [642]:
# Calculate percentage for all columns
missing_pct = (df_listings_clean.isna().sum() / len(df_listings_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

ClosePrice                  73.572035
BuyerOfficeName             72.073335
CloseDate                   70.862279
SubdivisionName             66.621900
PurchaseContractDate        57.570673
MainLevelBedrooms           53.776768
HighSchoolDistrict          39.701649
AttachedGarageYN            34.031233
AssociationFee              32.304032
Stories                     25.928510
Levels                      19.212589
GarageSpaces                18.906353
FireplaceYN                 14.282674
NewConstructionYN           13.384238
Latitude                    12.595909
Latitude.1                  12.595909
Longitude.1                 12.514589
Longitude                   12.514589
LivingArea                  12.404259
LivingArea.1                12.404259
PropertySubType             12.132074
BedroomsTotal               12.007742
MLSAreaMajor                11.326047
LotSizeAcres                 9.514166
LotSizeSquareFeet            9.198633
LotSizeArea                  9.074525
ListAgentEma

In [643]:
num_cols_with_missing = df_listings_clean.isna().any().sum()

print(f"Number of variables with missing values: {num_cols_with_missing}")

Number of variables with missing values: 46


In [644]:
cols_rows_dropping = missing_pct[(missing_pct < 1) & (missing_pct > 0)]
cols_rows_dropping

OriginalListPrice           0.385204
ListAgentFirstName          0.589511
ListAgentLastName           0.008625
UnparsedAddress             0.264457
ListPrice                   0.248439
ListAgentFullName           0.028003
CountyOrParish              0.000112
ListAgentFirstName.1        0.589511
StreetNumberNumeric         0.451514
City                        0.109434
ContractStatusChangeDate    0.825069
ListPrice.1                 0.248439
StateOrProvince             0.007953
ListAgentLastName.1         0.008625
PostalCode                  0.024418
UnparsedAddress.1           0.264457
dtype: float64

In [645]:
cols_rows_dropped = ['OriginalListPrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'CountyOrParish',
        'StreetNumberNumeric', 'City', 'ContractStatusChangeDate','StateOrProvince', 'PostalCode']

In [646]:
df_listings_clean = df_listings_clean.dropna(subset=['OriginalListPrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'CountyOrParish',
        'StreetNumberNumeric', 'City', 'ContractStatusChangeDate','StateOrProvince', 'PostalCode'])

In [647]:
df_listings_clean.shape

(871642, 57)

In [648]:
# Create a reference mapping of Names -> Emails
# Drop any rows that have missing emails so the map is 'clean'
email_lookup = df_listings_clean.dropna(subset=['ListAgentEmail']).drop_duplicates(['ListAgentFirstName', 'ListAgentLastName'])

# Set the names as the index to make the 'lookup' possible
email_lookup = email_lookup.set_index(['ListAgentFirstName', 'ListAgentLastName'])['ListAgentEmail']

# Create a function to apply the logic
def fill_missing_emails(row):
    # Only try to fill if the current email is missing
    if pd.isna(row['ListAgentEmail']):
        # Look up the name in our reference map
        return email_lookup.get((row['ListAgentFirstName'], row['ListAgentLastName']), "None")
    return row['ListAgentEmail']

# Apply it to the dataframe
df_listings_clean['ListAgentEmail'] = df_listings_clean.apply(fill_missing_emails, axis=1)

In [649]:
df_listings_clean['ListAgentEmail'].isna().sum()

np.int64(0)

In [650]:
(df_listings_clean['ListAgentEmail'] == 'None').sum()

np.int64(2857)

From the code above, I noticed that there are no missing values for list agent's first and last names, so to determine their emails, I tried to look up any matches by full name to get the list agent's emails for the rows that are missing them. If there are no matches, those emails will be preserved by having a value of "None" (i.e. no email provided).

In [651]:
df_listings_clean.isna().sum()

OriginalListPrice                0
ListingKey                       0
ListAgentEmail                   0
CloseDate                   616377
ClosePrice                  639081
ListAgentFirstName               0
ListAgentLastName                0
Latitude                    108864
Longitude                   108165
UnparsedAddress                  0
PropertyType                     0
LivingArea                  105011
ListPrice                        0
DaysOnMarket                     0
ListOfficeName                   0
BuyerOfficeName             625568
ListAgentFullName                0
ListingKeyNumeric                0
MLSAreaMajor                 98462
CountyOrParish                   0
PropertyType.1                   0
MlsStatus                        0
ListAgentFirstName.1             0
AttachedGarageYN            295408
ParkingTotal                 51673
PropertySubType             104627
LotSizeAcres                 82729
SubdivisionName             580451
YearBuilt           

In [652]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'PropertyType.1', 'MlsStatus', 'ListAgentFirstName.1',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'YearBuilt', 'DaysOnMarket.1', 'StreetNumberNumeric',
       'LivingArea.1', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'ContractStatusChangeDate', 'Longitude.1',
       'PurchaseContractDate', 'ListingContractDate', 'Latitude.1',
       'ListPrice.1', 'StateOrProvince', 'FireplaceYN', 'Stories', 'Levels',
       'ListAgentLastName.1', 'LotSizeArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDist

#### **Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)**

In [653]:
df_listings_clean['CloseDate'] = pd.to_datetime(df_listings_clean['CloseDate'], errors='coerce')

In [654]:
df_listings_clean['PurchaseContractDate'] = pd.to_datetime(df_listings_clean['PurchaseContractDate'], errors='coerce')

In [655]:
df_listings_clean['ListingContractDate'] = pd.to_datetime(df_listings_clean['ListingContractDate'], errors='coerce')

In [656]:
df_listings_clean['ContractStatusChangeDate'] = pd.to_datetime(df_listings_clean['ContractStatusChangeDate'], errors='coerce')

In [657]:
print(df_listings_clean['PurchaseContractDate'].dtype)

datetime64[us]


#### **Remove unnecessary or redundant columns**

In [658]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'PropertyType.1', 'MlsStatus', 'ListAgentFirstName.1',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'YearBuilt', 'DaysOnMarket.1', 'StreetNumberNumeric',
       'LivingArea.1', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'ContractStatusChangeDate', 'Longitude.1',
       'PurchaseContractDate', 'ListingContractDate', 'Latitude.1',
       'ListPrice.1', 'StateOrProvince', 'FireplaceYN', 'Stories', 'Levels',
       'ListAgentLastName.1', 'LotSizeArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDist

In [659]:
df_listings_clean.shape

(871642, 57)

In [660]:
df_listings_clean = df_listings_clean.drop(columns=['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 
                                                        'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'UnparsedAddress.1'])

In [661]:
df_listings_clean.shape

(871642, 48)

In [662]:
(df_listings_clean['OriginalListPrice'] == df_listings_clean['ListPrice']).value_counts()

True     703837
False    167805
Name: count, dtype: int64

#### **Outlier Cleaning**

Remove or flag invalid numeric values: ClosePrice <=0, LivingArea <=0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

Removing and inspecting negative values

In [663]:
neg_counts = (df_listings_clean.select_dtypes(include='number') < 0).sum()

In [664]:
neg_counts[neg_counts > 0]

Latitude             7
Longitude       763254
DaysOnMarket        37
ParkingTotal       173
dtype: int64

In [665]:
(neg_counts[neg_counts > 0] / df_listings_clean.shape[0] * 100).sort_values(ascending=False)

Longitude       87.565078
ParkingTotal     0.019848
DaysOnMarket     0.004245
Latitude         0.000803
dtype: float64

ParkingTotal        

In [666]:
neg_ParkingTotal = df_listings_clean[df_listings_clean['ParkingTotal'] < 0]
neg_ParkingTotal.head()

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
8443,2300000.0,1058688466,mabroukhamza093@gmail.com,NaT,NaN,Hamza,Mabrouk,NaN,NaN,725 Hyde Street,...,NaN,1650.0,NaN,NaN,NaN,San Francisco Unified,94109,0.0,1650.0,2024
12223,1258000.0,1058416008,trisha@trishamotter.com,2024-02-15,1600000.0,Trisha,Motter,NaN,NaN,7385 Forsum Road,...,NaN,5227.0,NaN,False,2.0,Other,95138,NaN,5227.0,2024
26306,1588888.0,1054056032,maryoproperties@yahoo.com,2024-02-23,1550000.0,Mary,O'neill,NaN,NaN,2965 Calle De Las Estrella,...,NaN,3920.0,NaN,False,2.0,Other,95148,120.0,3920.0,2024
30751,1599000.0,1061806243,kroyer.re@gmail.com,2024-04-02,1620000.0,Kathy,Royer,NaN,NaN,6088 Pietz Court,...,NaN,6098.0,NaN,False,2.0,San Jose Unified,95123,NaN,6098.0,2024
31804,1400000.0,1061770369,rick.talmage@compass.com,2024-03-27,1550000.0,Rick,Talmage,NaN,NaN,994 La Mesa Terrace E,...,NaN,NaN,NaN,False,2.0,Other,94086,655.0,NaN,2024


In [667]:
neg_ParkingTotal.head()

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
8443,2300000.0,1058688466,mabroukhamza093@gmail.com,NaT,NaN,Hamza,Mabrouk,NaN,NaN,725 Hyde Street,...,NaN,1650.0,NaN,NaN,NaN,San Francisco Unified,94109,0.0,1650.0,2024
12223,1258000.0,1058416008,trisha@trishamotter.com,2024-02-15,1600000.0,Trisha,Motter,NaN,NaN,7385 Forsum Road,...,NaN,5227.0,NaN,False,2.0,Other,95138,NaN,5227.0,2024
26306,1588888.0,1054056032,maryoproperties@yahoo.com,2024-02-23,1550000.0,Mary,O'neill,NaN,NaN,2965 Calle De Las Estrella,...,NaN,3920.0,NaN,False,2.0,Other,95148,120.0,3920.0,2024
30751,1599000.0,1061806243,kroyer.re@gmail.com,2024-04-02,1620000.0,Kathy,Royer,NaN,NaN,6088 Pietz Court,...,NaN,6098.0,NaN,False,2.0,San Jose Unified,95123,NaN,6098.0,2024
31804,1400000.0,1061770369,rick.talmage@compass.com,2024-03-27,1550000.0,Rick,Talmage,NaN,NaN,994 La Mesa Terrace E,...,NaN,NaN,NaN,False,2.0,Other,94086,655.0,NaN,2024


In [668]:
df_listings_clean['ParkingTotal'].describe()

count    8.199690e+05
mean     6.342753e+00
std      2.873782e+03
min     -1.430000e+02
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      2.593669e+06
Name: ParkingTotal, dtype: float64

DaysOnMarket              

In [669]:
neg_DaysOnMarket = df_listings_clean[df_listings_clean['DaysOnMarket'] < 0]
neg_DaysOnMarket.head()

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
223,799000.0,1063549350,ainsleyhughes@kw.com,2024-03-21,799000.0,Ainsley,Hughes,34.425577,-119.291855,11905 Silver Spur Street,...,One,5885.0,NaN,NaN,2.0,NaN,93023,59.0,5885.0,2024
228,899000.0,1063528331,absea@comcast.net,2024-03-19,810000.0,A.B.,Priceman DRE 0126...,39.383806,-123.789254,31530 Emerald Drive,...,One,0.8,NaN,False,2.0,NaN,95437,NaN,34848.0,2024
752,11500.0,1061271257,teresa.fuller@compass.com,2024-03-07,10500.0,Teresa,Fuller,34.081125,-118.363403,536 N Edinburgh Avenue,...,One,6612.0,5.0,NaN,0.0,NaN,90048,0.0,6612.0,2024
1119,1599000.0,1060153479,robert@anppros.com,2024-03-21,1625000.0,Robert,Perez,34.244449,-118.265167,3929 El Moreno Street,...,Two,5249.0,NaN,NaN,2.0,NaN,91214,NaN,5249.0,2024
4167,469999.0,1059512539,khoren9@yahoo.com,NaT,NaN,Khoren,Barutyan,34.201700,-118.460053,15015 Sherman Way 103,...,"One,Two",15574.0,0.0,False,2.0,Los Angeles Unified,91405,390.0,15574.0,2024


In [670]:
rows_DaysOnMarket = neg_DaysOnMarket['DaysOnMarket']
rows_DaysOnMarket.head()

223    -48
228    -58
752    -16
1119    -1
4167   -33
Name: DaysOnMarket, dtype: int64

In [671]:
non_neg = (df_listings_clean['DaysOnMarket'] >= 0) & (df_listings_clean['ParkingTotal'] >= 0)

In [672]:
df_listings_clean = df_listings_clean[non_neg].copy()

In [673]:
df_listings_clean.shape

(819760, 48)

In [674]:
neg_counts = (df_listings_clean.select_dtypes(include='number') < 0).sum()

In [675]:
neg_counts[neg_counts > 0]

Latitude          4
Longitude    716097
dtype: int64

In [676]:
df_listings_clean['missing_coordinates_flag'] = df_listings_clean['Latitude'].isna() | df_listings_clean['Longitude'].isna()

In [677]:
df_listings_clean['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    715565
True     104195
Name: count, dtype: int64

In [678]:
df_listings_clean['sentinel_null_flag'] = (df_listings_clean['Latitude'] == 0) | (df_listings_clean['Longitude'] == 0)

In [679]:
df_listings_clean['sentinel_null_flag'].value_counts()

sentinel_null_flag
False    819675
True         85
Name: count, dtype: int64

In [680]:
df_listings_clean['positive_longitude_flag'] = df_listings_clean['Longitude'] > 0

In [681]:
df_listings_clean['positive_longitude_flag'].value_counts()

positive_longitude_flag
False    819675
True         85
Name: count, dtype: int64

In [682]:
df_listings_clean['out_of_state_flag'] = (
    ~(
        (df_listings_clean['Latitude'] >= 32.5) &
        (df_listings_clean['Latitude'] <= 42.0) &
        (df_listings_clean['Longitude'] >= -124.5) &
        (df_listings_clean['Longitude'] <= -114.1)
    )
    & ~df_listings_clean['missing_coordinates_flag']
    & ~df_listings_clean['sentinel_null_flag']
)

In [683]:
df_listings_clean['out_of_state_flag'].value_counts()

out_of_state_flag
False    819495
True        265
Name: count, dtype: int64

In [684]:
df_listings_clean.shape

(819760, 52)

In [685]:
'''
df_map = df_listings_clean.sample(30000)

fig = px.scatter_mapbox(
    df_map,
    lat="Latitude",
    lon="Longitude",
    color="OriginalListPrice",
    size_max=15,
    zoom=5,
    title="California Property Value Distribution",
    color_continuous_scale=px.colors.sequential.Plasma,
    mapbox_style="open-street-map"
)

fig.show(renderer='browser')
'''

'\ndf_map = df_listings_clean.sample(30000)\n\nfig = px.scatter_mapbox(\n    df_map,\n    lat="Latitude",\n    lon="Longitude",\n    color="OriginalListPrice",\n    size_max=15,\n    zoom=5,\n    title="California Property Value Distribution",\n    color_continuous_scale=px.colors.sequential.Plasma,\n    mapbox_style="open-street-map"\n)\n\nfig.show(renderer=\'browser\')\n'

#### **Further Handling Missing Values Appropriately**

In [686]:
missing_listing_counts = df_listings_clean.isnull().sum()
missing_listing_counts

OriginalListPrice                0
ListingKey                       0
ListAgentEmail                   0
CloseDate                   568819
ClosePrice                  591486
ListAgentFirstName               0
ListAgentLastName                0
Latitude                    104195
Longitude                   103493
UnparsedAddress                  0
PropertyType                     0
LivingArea                   53333
ListPrice                        0
DaysOnMarket                     0
ListOfficeName                   0
BuyerOfficeName             577961
ListAgentFullName                0
ListingKeyNumeric                0
MLSAreaMajor                 92576
CountyOrParish                   0
MlsStatus                        0
AttachedGarageYN            243713
ParkingTotal                     0
PropertySubType              62050
LotSizeAcres                 81382
SubdivisionName             533314
YearBuilt                    24503
StreetNumberNumeric              0
ListingId           

In [687]:
missing_listing_percent = (df_listings_clean.isnull().mean()) * 100
missing_listing_percent

OriginalListPrice            0.000000
ListingKey                   0.000000
ListAgentEmail               0.000000
CloseDate                   69.388480
ClosePrice                  72.153557
ListAgentFirstName           0.000000
ListAgentLastName            0.000000
Latitude                    12.710427
Longitude                   12.624793
UnparsedAddress              0.000000
PropertyType                 0.000000
LivingArea                   6.505929
ListPrice                    0.000000
DaysOnMarket                 0.000000
ListOfficeName               0.000000
BuyerOfficeName             70.503684
ListAgentFullName            0.000000
ListingKeyNumeric            0.000000
MLSAreaMajor                11.293061
CountyOrParish               0.000000
MlsStatus                    0.000000
AttachedGarageYN            29.729799
ParkingTotal                 0.000000
PropertySubType              7.569289
LotSizeAcres                 9.927540
SubdivisionName             65.057334
YearBuilt   

In [688]:
missing_summary = pd.DataFrame({
    "missing_listing_counts": missing_listing_counts,
    "missing_listing_percent": missing_listing_percent
})

In [689]:
missing_listing_summary = missing_summary.sort_values(by="missing_listing_percent", ascending=False)
print(missing_listing_summary)

                          missing_listing_counts  missing_listing_percent
ClosePrice                                591486                72.153557
BuyerOfficeName                           577961                70.503684
CloseDate                                 568819                69.388480
SubdivisionName                           533314                65.057334
PurchaseContractDate                      453293                55.295818
MainLevelBedrooms                         407939                49.763223
HighSchoolDistrict                        289918                35.366205
AssociationFee                            263650                32.161852
AttachedGarageYN                          243713                29.729799
Stories                                   166869                20.355836
GarageSpaces                              110931                13.532131
Levels                                    107349                13.095174
Latitude                              

In [690]:
missing_listing_summary[missing_listing_summary['missing_listing_percent'] > 0]

,missing_listing_counts,missing_listing_percent
ClosePrice,591486,72.153557
BuyerOfficeName,577961,70.503684
CloseDate,568819,69.388480
SubdivisionName,533314,65.057334
PurchaseContractDate,453293,55.295818
MainLevelBedrooms,407939,49.763223
HighSchoolDistrict,289918,35.366205
AssociationFee,263650,32.161852
AttachedGarageYN,243713,29.729799
Stories,166869,20.355836


In [691]:
continuous_columns = df_listings_clean.select_dtypes(include=['int64', 'float64']).columns

In [692]:
#df_listings_clean[continuous_columns].corr(method = "spearman").round(2)

In [693]:
'''corrs =df_listings_clean[continuous_columns].corr(method = "spearman")
mask = np.zeros_like(corrs)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corrs, cmap='Spectral_r', mask=mask, square=True, vmin=-.4, vmax=.4)
plt.title('Correlation matrix')'''

'corrs =df_listings_clean[continuous_columns].corr(method = "spearman")\nmask = np.zeros_like(corrs)\nmask[np.triu_indices_from(mask)] = True\nsns.heatmap(corrs, cmap=\'Spectral_r\', mask=mask, square=True, vmin=-.4, vmax=.4)\nplt.title(\'Correlation matrix\')'

In [694]:
'''pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)'''

"pd.set_option('display.max_rows', None)\npd.set_option('display.max_columns', None)\npd.set_option('display.width', None)\npd.set_option('display.max_colwidth', None)"

In [695]:
'''corr_matrix = df_listings_clean[continuous_columns].corr(method="spearman").round(2)

moderate_corr = corr_matrix.stack()
moderate_corr = moderate_corr[(moderate_corr.abs() >= 0.3) & (moderate_corr.abs() < 0.7)]

print(moderate_corr)'''

'corr_matrix = df_listings_clean[continuous_columns].corr(method="spearman").round(2)\n\nmoderate_corr = corr_matrix.stack()\nmoderate_corr = moderate_corr[(moderate_corr.abs() >= 0.3) & (moderate_corr.abs() < 0.7)]\n\nprint(moderate_corr)'

- Under 10% missing -> simple statistical imputation method, but for categorical variables, may include a "Missing" category if deemed necessary
- 10-50% missing -> "Decision Zone" - meaning that there are different approaches that can be taken to impute the missing values like a group-based imputation, model-based if justifiable, adding a missing indicator, or doing simple imputation if necessary
- above 50% -> Since at this point, they are probably important features, the missingness in the data could be important and signals themselves. 

Let's start with "Under 10% missing"

In [696]:
under_10 = missing_listing_summary[(missing_listing_summary['missing_listing_percent'] < 10) & (missing_listing_percent != 0)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_13868\2293341584.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  under_10 = missing_listing_summary[(missing_listing_summary['missing_listing_percent'] < 10) & (missing_listing_percent != 0)]


In [697]:
under_10

,missing_listing_counts,missing_listing_percent
LotSizeAcres,81382,9.927540
LotSizeSquareFeet,78684,9.598419
LotSizeArea,77666,9.474236
FireplaceYN,69549,8.484069
PropertySubType,62050,7.569289
LivingArea,53333,6.505929
BedroomsTotal,49617,6.052625
YearBuilt,24503,2.989046
BathroomsTotalInteger,18187,2.218576


In [698]:
df_listings_clean['MLSAreaMajor'].describe()

count                727184
unique                 1142
top       699 - Not Defined
freq                  77778
Name: MLSAreaMajor, dtype: object

In [699]:
df_listings_clean['MLSAreaMajor'].value_counts().head()

MLSAreaMajor
699 - Not Defined                     77778
SRCAR - Southwest Riverside County    31291
252 - Riverside                        8436
C14 - Santa Monica                     5799
C42 - Downtown L.A.                    5576
Name: count, dtype: int64

In [700]:
df_listings_clean['MLSAreaMajor'].value_counts(normalize=True).head()

MLSAreaMajor
699 - Not Defined                     0.106958
SRCAR - Southwest Riverside County    0.043030
252 - Riverside                       0.011601
C14 - Santa Monica                    0.007975
C42 - Downtown L.A.                   0.007668
Name: proportion, dtype: float64

In [701]:
df_listings_clean['MLSAreaMajor'] = df_listings_clean['MLSAreaMajor'].fillna("Missing")

In [702]:
df_listings_clean['MLSAreaMajor'].isna().sum()

np.int64(0)

In [703]:
df_listings_clean['LotSizeAcres'].describe()

count    7.383780e+05
mean     5.675789e+02
std      4.310205e+05
min      0.000000e+00
25%      1.194000e-01
50%      1.700000e-01
75%      3.200000e-01
max      3.702600e+08
Name: LotSizeAcres, dtype: float64

In [704]:
df_listings_clean['LotSizeAcres'] = df_listings_clean['LotSizeAcres'].fillna(df_listings_clean['LotSizeAcres'].median())

In [705]:
df_listings_clean['LotSizeAcres'].isna().sum()

np.int64(0)

The mean is greater than the median. The values are skewed to the right. I will be using the median to impute. 

In [706]:
df_listings_clean['LotSizeArea'].describe()

count    7.420940e+05
mean     5.631998e+04
std      2.438367e+06
min      0.000000e+00
25%      4.934000e+03
50%      7.129000e+03
75%      1.220000e+04
max      9.187423e+08
Name: LotSizeArea, dtype: float64

In [707]:
df_listings_clean['LotSizeArea'] = df_listings_clean['LotSizeArea'].fillna(df_listings_clean['LotSizeArea'].median())

In [708]:
df_listings_clean['LotSizeArea'].isna().sum()

np.int64(0)

In [709]:
df_listings_clean['LotSizeArea'].describe()

count    8.197600e+05
mean     5.165951e+04
std      2.320030e+06
min      0.000000e+00
25%      5.150000e+03
50%      7.129000e+03
75%      1.096900e+04
max      9.187423e+08
Name: LotSizeArea, dtype: float64

In [710]:
df_listings_clean['LotSizeSquareFeet'].describe()

count    7.410760e+05
mean     5.282047e+05
std      2.247518e+07
min      0.000000e+00
25%      5.180000e+03
50%      7.380000e+03
75%      1.386900e+04
max      7.638115e+09
Name: LotSizeSquareFeet, dtype: float64

The mean is less than the median. The values are skewed to the left. I will be using the median to impute the missing values. 

In [711]:
df_listings_clean['LotSizeSquareFeet'] = df_listings_clean['LotSizeSquareFeet'].fillna(df_listings_clean['LotSizeSquareFeet'].median())

In [712]:
df_listings_clean['LotSizeSquareFeet'].isna().sum()

np.int64(0)

In [713]:
df_listings_clean['LotSizeSquareFeet'].describe()

count    8.197600e+05
mean     4.782137e+05
std      2.136989e+07
min      0.000000e+00
25%      5.420000e+03
50%      7.380000e+03
75%      1.203400e+04
max      7.638115e+09
Name: LotSizeSquareFeet, dtype: float64

In [714]:
df_listings_clean['FireplaceYN'].value_counts()

FireplaceYN
True     449622
False    300589
Name: count, dtype: int64

In [715]:
df_listings_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.599327
False    0.400673
Name: proportion, dtype: float64

In [716]:
df_listings_clean['FireplaceYN'] = df_listings_clean['FireplaceYN'].fillna(df_listings_clean['FireplaceYN'].mode()[0])

In [717]:
df_listings_clean['FireplaceYN'].value_counts()

FireplaceYN
True     519171
False    300589
Name: count, dtype: int64

In [718]:
df_listings_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.633321
False    0.366679
Name: proportion, dtype: float64

In [719]:
df_listings_clean['FireplaceYN'].isna().sum()

np.int64(0)

In [720]:
df_listings_clean['PropertySubType'].value_counts().head()

PropertySubType
SingleFamilyResidence    490126
Condominium              151530
Townhouse                 45972
Apartment                 20207
Duplex                    17176
Name: count, dtype: int64

In [721]:
df_listings_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.646852
Condominium              0.199984
Townhouse                0.060672
Apartment                0.026669
Duplex                   0.022668
Name: proportion, dtype: float64

In [722]:
df_listings_clean['PropertySubType'] = df_listings_clean['PropertySubType'].fillna(df_listings_clean['PropertySubType'].mode()[0])

In [723]:
df_listings_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.673583
Condominium              0.184847
Townhouse                0.056080
Apartment                0.024650
Duplex                   0.020952
Name: proportion, dtype: float64

In [724]:
df_listings_clean['LivingArea'].describe()

count    7.664270e+05
mean     3.126496e+03
std      1.038604e+06
min      0.000000e+00
25%      1.180000e+03
50%      1.600000e+03
75%      2.236000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

The mean is greater than the median, which shows that it is skewing to the right. I will then impute by the median rather than the mean.

In [725]:
df_listings_clean['LivingArea'] = df_listings_clean['LivingArea'].fillna(df_listings_clean['LivingArea'].median())

In [726]:
df_listings_clean['LivingArea'].describe()

count    8.197600e+05
mean     3.027184e+03
std      1.004250e+06
min      0.000000e+00
25%      1.206000e+03
50%      1.600000e+03
75%      2.172000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

In [727]:
df_listings_clean['LivingArea'].isna().sum()

np.int64(0)

In [728]:
df_listings_clean['BedroomsTotal'].value_counts().head()

BedroomsTotal
3.0    272629
2.0    181983
4.0    177920
5.0     58222
1.0     49635
Name: count, dtype: int64

In [729]:
df_listings_clean['BedroomsTotal'].value_counts(normalize=True).head()

BedroomsTotal
3.0    0.353998
2.0    0.236298
4.0    0.231022
5.0    0.075599
1.0    0.064449
Name: proportion, dtype: float64

Will add a missing indicator for the BedroomsTotal variable to see if the missingness has a real signal moving forward, but will impute based on the median

In [730]:
df_listings_clean['BedroomsTotal_Missing'] = df_listings_clean['BedroomsTotal'].isna().astype(int)

In [731]:
df_listings_clean['BedroomsTotal'] = df_listings_clean['BedroomsTotal'].fillna(df_listings_clean['BedroomsTotal'].median())

In [732]:
df_listings_clean.shape

(819760, 53)

In [733]:
df_listings_clean['BedroomsTotal_Missing'].value_counts()

BedroomsTotal_Missing
0    770143
1     49617
Name: count, dtype: int64

In [734]:
df_listings_clean['BedroomsTotal'].value_counts(normalize=True).head()

BedroomsTotal
3.0    0.393098
2.0    0.221995
4.0    0.217039
5.0    0.071023
1.0    0.060548
Name: proportion, dtype: float64

Will do the same for YearBuilt

In [735]:
df_listings_clean['YearBuilt'].describe()

count    795257.000000
mean       1978.580420
std          28.065725
min        1776.000000
25%        1960.000000
50%        1979.000000
75%        2001.000000
max        2028.000000
Name: YearBuilt, dtype: float64

In [736]:
df_listings_clean['YearBuilt_Missing'] = df_listings_clean['YearBuilt'].isna().astype(int)

In [737]:
df_listings_clean['YearBuilt'] = df_listings_clean['YearBuilt'].fillna(round(df_listings_clean['YearBuilt'].mean()))

In [738]:
(df_listings_clean['YearBuilt'] % 1 == 0).value_counts()

YearBuilt
True    819760
Name: count, dtype: int64

As well as BathroomsTotalInteger

In [739]:
df_listings_clean['BathroomsTotalInteger'].describe()

count    801573.000000
mean          2.507720
std           3.011914
min           0.000000
25%           2.000000
50%           2.000000
75%           3.000000
max        2208.000000
Name: BathroomsTotalInteger, dtype: float64

In [740]:
df_listings_clean['BathroomsTotalInteger'] = df_listings_clean['BathroomsTotalInteger'].fillna(round(df_listings_clean['BathroomsTotalInteger'].mean()))

In [741]:
df_listings_clean['BathroomsTotalInteger'].value_counts(normalize=True).head()

BathroomsTotalInteger
2.0    0.380825
3.0    0.307428
1.0    0.136306
4.0    0.076333
0.0    0.039430
Name: proportion, dtype: float64

Onto the 10-50% missing

In [742]:
btwn_10_50 = missing_listing_summary[(missing_listing_summary['missing_listing_percent'] >= 10) & (missing_listing_percent <= 50)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_13868\3278696572.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  btwn_10_50 = missing_listing_summary[(missing_listing_summary['missing_listing_percent'] >= 10) & (missing_listing_percent <= 50)]


In [743]:
btwn_10_50

,missing_listing_counts,missing_listing_percent
MainLevelBedrooms,407939,49.763223
HighSchoolDistrict,289918,35.366205
AssociationFee,263650,32.161852
AttachedGarageYN,243713,29.729799
Stories,166869,20.355836
GarageSpaces,110931,13.532131
Levels,107349,13.095174
Latitude,104195,12.710427
Longitude,103493,12.624793
NewConstructionYN,99355,12.120011


In [744]:
df_listings_clean['MainLevelBedrooms'].describe()

count    411821.000000
mean          1.935754
std           5.966382
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max        3333.000000
Name: MainLevelBedrooms, dtype: float64

In [745]:
df_listings_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
1.0    0.258809
3.0    0.240995
2.0    0.194997
0.0    0.177485
4.0    0.105594
Name: proportion, dtype: float64

In [746]:
df_listings_clean['MainLevelBedrooms_Missing'] = df_listings_clean['MainLevelBedrooms'].isna().astype(int)

In [747]:
df_listings_clean['MainLevelBedrooms'] = df_listings_clean['MainLevelBedrooms'].fillna(df_listings_clean['MainLevelBedrooms'].median())

In [748]:
df_listings_clean['MainLevelBedrooms'].describe()

count    819760.000000
mean          1.967725
std           4.228966
min           0.000000
25%           2.000000
50%           2.000000
75%           2.000000
max        3333.000000
Name: MainLevelBedrooms, dtype: float64

In [749]:
df_listings_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
2.0    0.595593
1.0    0.130017
3.0    0.121068
0.0    0.089163
4.0    0.053047
Name: proportion, dtype: float64

The mean and the median is about the same, so using mean to impute the missing values but also add a missing indicator.

In [750]:
df_listings_clean['MainLevelBedrooms'].isna().sum()

np.int64(0)

In [751]:
df_listings_clean['HighSchoolDistrict'].value_counts().head()

HighSchoolDistrict
Los Angeles Unified          56106
Other                        52710
Capistrano Unified           17201
Irvine Unified               12168
Saddleback Valley Unified     9879
Name: count, dtype: int64

In [752]:
df_listings_clean['HighSchoolDistrict'].value_counts(normalize=True).head()

HighSchoolDistrict
Los Angeles Unified          0.105892
Other                        0.099482
Capistrano Unified           0.032464
Irvine Unified               0.022965
Saddleback Valley Unified    0.018645
Name: proportion, dtype: float64

In [753]:
df_listings_clean['HighSchoolDistrict'] = df_listings_clean['HighSchoolDistrict'].fillna("Missing")

In [754]:
df_listings_clean['HighSchoolDistrict'].value_counts()

HighSchoolDistrict
Missing                289918
Los Angeles Unified     56106
Other                   52710
Capistrano Unified      17201
Irvine Unified          12168
                        ...  
Clear Creek                 1
Wishon                      1
Reeds Creek                 1
Manzanita                   1
Sunol Glen                  1
Name: count, Length: 458, dtype: int64

In [755]:
df_listings_clean['HighSchoolDistrict'].isna().sum()

np.int64(0)

In [756]:
df_listings_clean['AssociationFee'].describe()

count    556110.000000
mean        216.111424
std        2101.397649
min           0.000000
25%           0.000000
50%           0.000000
75%         331.882500
max      968348.000000
Name: AssociationFee, dtype: float64

In [757]:
# Regression Predictions
def impute_lgbm_numeric(df, target):
    df = df.copy()

    # mask missing values
    missing_mask = df[target].isna()

    if missing_mask.sum() == 0:
        return df

    # split data
    train = df[~missing_mask]
    test = df[missing_mask]

    y_train = train[target]
    X_train = train.drop(columns=[target]).select_dtypes(include='number')
    X_test = test.drop(columns=[target]).select_dtypes(include='number')

    # encode categorical columns safely
    for col in X_train.select_dtypes(include=['object', 'category']).columns:
        X_train[col] = X_train[col].astype('category').cat.codes
        X_test[col] = X_test[col].astype('category').cat.codes

    # model
    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # constraint: association_fee cannot be negative
    preds = np.clip(preds, a_min=0, a_max=None)

    df.loc[missing_mask, target] = preds

    return df

In [758]:
df_listings_clean = impute_lgbm_numeric(df_listings_clean, "AssociationFee")

In [759]:
df_listings_clean['AssociationFee'].describe()

count    819760.000000
mean        232.029305
std        1755.696078
min           0.000000
25%           0.000000
50%         100.000000
75%         308.263928
max      968348.000000
Name: AssociationFee, dtype: float64

In [760]:
df_listings_clean['AssociationFee'].isna().sum()

np.int64(0)

In [761]:
(df_listings_clean['AssociationFee'] < 0).value_counts()

AssociationFee
False    819760
Name: count, dtype: int64

In [762]:
df_listings_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True     444063
False    131984
Name: count, dtype: int64

In [763]:
df_listings_clean['AttachedGarageYN'] = df_listings_clean['AttachedGarageYN'].fillna("Missing")

In [764]:
df_listings_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True       444063
Missing    243713
False      131984
Name: count, dtype: int64

In [765]:
df_listings_clean['Stories'].describe()

count    652891.000000
mean          1.369248
std           0.482602
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           2.000000
Name: Stories, dtype: float64

In [766]:
df_listings_clean['Stories'].value_counts()

Stories
1.0    411812
2.0    241079
Name: count, dtype: int64

In [767]:
df_listings_clean['Stories'].value_counts(normalize=True)

Stories
1.0    0.630752
2.0    0.369248
Name: proportion, dtype: float64

In [768]:
df_listings_clean['Stories_Missing'] = df_listings_clean['Stories'].isna().astype(int)

In [769]:
df_listings_clean['Stories'] = df_listings_clean['Stories'].fillna(df_listings_clean['Stories'].median())

In [770]:
df_listings_clean['Stories'].value_counts()

Stories
1.0    578681
2.0    241079
Name: count, dtype: int64

In [771]:
df_listings_clean['Stories'].isna().sum()

np.int64(0)

In [772]:
df_listings_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    399896
0.0    125620
1.0     88779
3.0     74191
4.0     12300
Name: count, dtype: int64

In [773]:
df_listings_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.564164
0.0    0.177222
1.0    0.125247
3.0    0.104667
4.0    0.017353
Name: proportion, dtype: float64

In [774]:
df_listings_clean['GarageSpaces'].unique()

array([  2.  ,    nan,   3.  ,   4.  ,   0.  ,   1.  ,   6.  ,   7.  ,
         2.5 ,   3.5 ,   8.  ,   5.  ,  13.  ,   9.  ,  20.  ,  14.  ,
        25.  ,  10.  , 400.  ,  16.  ,  12.  ,  52.  ,   1.5 ,  26.  ,
       200.  ,  18.  ,   4.5 ,  22.  , 500.  ,  30.  ,  11.  ,  21.  ,
        34.  ,  15.  ,  82.  ,  19.  ,  35.  ,  80.  ,  49.  ,  40.  ,
        98.  ,  24.  ,  54.  ,  50.  ,  44.  , 300.  ,  60.  ,  17.  ,
        84.  ,  48.  ,  83.  ,  33.  , 600.  ,   5.5 ,  71.  ,  32.  ,
        78.  ,  29.  ,  28.  ,  23.  ,  86.  ,  72.  ,   6.5 ,  31.  ,
        27.  ,  36.  ,   8.5 ,  51.  ,   2.25,  45.  , 100.  ,  38.  ,
        46.  ,  75.  ,  41.  ,  55.  ,  64.  ,  88.  ,  96.  ,  37.  ])

In [775]:
df_listings_clean['GarageSpaces_Missing'] = df_listings_clean['GarageSpaces'].isna().astype(int)

In [776]:
df_listings_clean['GarageSpaces'] = df_listings_clean['GarageSpaces'].fillna(df_listings_clean['GarageSpaces'].median())

In [777]:
df_listings_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    510827
0.0    125620
1.0     88779
3.0     74191
4.0     12300
Name: count, dtype: int64

In [778]:
df_listings_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.623142
0.0    0.153240
1.0    0.108299
3.0    0.090503
4.0    0.015004
Name: proportion, dtype: float64

In [779]:
df_listings_clean['GarageSpaces'].isna().sum()

np.int64(0)

In [780]:
df_listings_clean['NewConstructionYN'].value_counts()

NewConstructionYN
False    688779
True      31626
Name: count, dtype: int64

In [781]:
df_listings_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.9561
True     0.0439
Name: proportion, dtype: float64

In [782]:
df_listings_clean['NewConstructionYN'] = df_listings_clean['NewConstructionYN'].fillna(df_listings_clean['NewConstructionYN'].mode()[0])

In [783]:
df_listings_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.96142
True     0.03858
Name: proportion, dtype: float64

In [784]:
df_listings_clean['NewConstructionYN'].isna().sum()

np.int64(0)

In [785]:
df_listings_clean['Levels'].value_counts().head()

Levels
One            406712
Two            239320
ThreeOrMore     38949
MultiSplit      20014
One,Two          2546
Name: count, dtype: int64

In [786]:
df_listings_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.570895
Two            0.335930
ThreeOrMore    0.054672
MultiSplit     0.028093
One,Two        0.003574
Name: proportion, dtype: float64

In [787]:
df_listings_clean['Levels'].unique()

<StringArray>
[                           'One',                            'Two',
                    'ThreeOrMore',                     'MultiSplit',
                              nan,                        'One,Two',
                'Two,ThreeOrMore',     'Two,ThreeOrMore,MultiSplit',
             'One,Two,MultiSplit',         'ThreeOrMore,MultiSplit',
                 'Two,MultiSplit',                'One,ThreeOrMore',
                 'One,MultiSplit',            'One,Two,ThreeOrMore',
     'One,ThreeOrMore,MultiSplit', 'One,Two,ThreeOrMore,MultiSplit',
                        'Two,One',                 'MultiSplit,One',
             'Two,MultiSplit,One',                'ThreeOrMore,One']
Length: 20, dtype: str

In [788]:
df_listings_clean['Levels'] = df_listings_clean['Levels'].fillna("Missing")

In [789]:
df_listings_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.496135
Two            0.291939
Missing        0.130952
ThreeOrMore    0.047513
MultiSplit     0.024414
Name: proportion, dtype: float64

In [790]:
df_listings_clean['Levels'].isna().sum()

np.int64(0)

Over 50% missing values

In [791]:
over_50 = missing_listing_summary[missing_listing_summary['missing_listing_percent'] >= 50]

In [792]:
over_50

,missing_listing_counts,missing_listing_percent
ClosePrice,591486,72.153557
BuyerOfficeName,577961,70.503684
CloseDate,568819,69.388480
SubdivisionName,533314,65.057334
PurchaseContractDate,453293,55.295818


In [793]:
df_listings_clean['ClosePrice'].describe()

count    2.282740e+05
mean     7.782772e+05
std      3.486934e+06
min      0.000000e+00
25%      4.950000e+03
50%      5.700000e+05
75%      1.020000e+06
max      8.200000e+08
Name: ClosePrice, dtype: float64

In [794]:
df_listings_clean["ClosePrice_Missing"] = df_listings_clean["ClosePrice"].isna().astype(int)

In [795]:
df_listings_clean['ClosePrice_Missing'].value_counts()

ClosePrice_Missing
1    591486
0    228274
Name: count, dtype: int64

In [796]:
df_listings_clean['ClosePrice'].describe()

count    2.282740e+05
mean     7.782772e+05
std      3.486934e+06
min      0.000000e+00
25%      4.950000e+03
50%      5.700000e+05
75%      1.020000e+06
max      8.200000e+08
Name: ClosePrice, dtype: float64

In [797]:
df_listings_clean['CloseDate'].value_counts()

CloseDate
2024-03-29    1330
2025-10-31    1301
2024-04-30    1300
2025-11-21    1285
2026-02-13    1259
              ... 
2026-12-21       1
2026-09-25       1
2026-05-17       1
2026-06-20       1
2026-07-22       1
Name: count, Length: 942, dtype: int64

In [798]:
missing_close_date = df_listings_clean[df_listings_clean['CloseDate'].isna()]

In [799]:
missing_close_date.shape

(568819, 58)

In [800]:
missing_close_date['PurchaseContractDate'].isna().sum()

np.int64(447077)

In [801]:
missing_close_date['PurchaseContractDate'].isna().sum() / missing_close_date.shape[0]

np.float64(0.7859740972084266)

About 79% of missing close price entries have missing purchase contract date

This could signal that the missingness in close price is associated with another missing variable which is purchase contract date. So, the data could not be completely missing at random. 

In [802]:
pd.crosstab(
    df_listings_clean['CloseDate'].isna(),
    df_listings_clean['PurchaseContractDate'].isna(),
    normalize='index'
)

PurchaseContractDate,False,True
CloseDate,,
False,0.975229,0.024771
True,0.214026,0.785974


In [803]:
print(df_listings_clean['PurchaseContractDate'].dtype)

datetime64[us]


In [804]:
print(df_listings_clean['CloseDate'].dtype)

datetime64[us]


In [805]:
df_listings_clean['No_Purchase_Date'] = df_listings_clean['PurchaseContractDate'].isna()
df_listings_clean['No_Close_Date'] = df_listings_clean['CloseDate'].isna()

In [806]:
df_listings_clean['PurchaseContractDate'].isna().sum()

np.int64(453293)

In [807]:
df_listings_clean['CloseDate'].value_counts(dropna=False).head()

CloseDate
NaT           568819
2024-03-29      1330
2025-10-31      1301
2024-04-30      1300
2025-11-21      1285
Name: count, dtype: int64

In [808]:
df_listings_clean['PurchaseContractDate'].value_counts(dropna=False).head()

PurchaseContractDate
NaT           453293
2024-05-01      1182
2024-04-24      1096
2024-04-29      1034
2024-04-30      1028
Name: count, dtype: int64

In [809]:
df_listings_clean['No_Purchase_Date'].value_counts()

No_Purchase_Date
True     453293
False    366467
Name: count, dtype: int64

In [810]:
df_listings_clean['No_Close_Date'].value_counts()

No_Close_Date
True     568819
False    250941
Name: count, dtype: int64

Keep the date variables with NaT to show those events were not observed or did not occur - I also created corresponding variables to measure this

In [811]:
df_listings_clean['SubdivisionName'].value_counts().head()

SubdivisionName
Other               8056
Not Applicable-1    7358
Not Applicable      3944
San Diego           3802
Not Listed          3725
Name: count, dtype: int64

In [812]:
df_listings_clean['SubdivisionName'] = df_listings_clean['SubdivisionName'].fillna("Missing")

In [813]:
df_listings_clean['SubdivisionName'].value_counts().head()

SubdivisionName
Missing             533314
Other                 8056
Not Applicable-1      7358
Not Applicable        3944
San Diego             3802
Name: count, dtype: int64

In [814]:
df_listings_clean.isna().sum().sum()

np.int64(2399247)

In [815]:
df_listings_clean[['ClosePrice', 'CloseDate', 'PurchaseContractDate']].isna().sum()

ClosePrice              591486
CloseDate               568819
PurchaseContractDate    453293
dtype: int64

In [816]:
df_listings_clean.shape

(819760, 60)

In [817]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'Year', 'missing_coordinates_flag',
       'sentinel_null_flag', 'positive_longitude_flag'

In [818]:
df_listings_clean.shape

(819760, 60)

In [819]:
df_listings_clean.isna().sum().sum()

np.int64(2399247)

In [820]:
df_listings_clean.isna().sum().sum() / df_listings_clean.notna().sum().sum()

np.float64(0.051280915184818955)

In [821]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'Year', 'missing_coordinates_flag',
       'sentinel_null_flag', 'positive_longitude_flag'

#### **Data Consistency Checks**

In [822]:
df_listings_clean['listing_after_close_flag'] = df_listings_clean['ListingContractDate'] > df_listings_clean['CloseDate']

In [823]:
df_listings_clean['listing_after_close_flag'].value_counts()

listing_after_close_flag
False    819621
True        139
Name: count, dtype: int64

In [824]:
df_listings_clean['purchase_after_close_flag'] = df_listings_clean['PurchaseContractDate'] > df_listings_clean['CloseDate']

In [825]:
df_listings_clean['purchase_after_close_flag'].value_counts()

purchase_after_close_flag
False    819645
True        115
Name: count, dtype: int64

In [826]:
df_listings_clean['negative_timeline_flag'] = (
   ( df_listings_clean['ListingContractDate'] > df_listings_clean['PurchaseContractDate']) |
   (df_listings_clean['PurchaseContractDate'] > df_listings_clean['CloseDate']) |
   (df_listings_clean['ListingContractDate'] > df_listings_clean['CloseDate'])
)

In [827]:
df_listings_clean['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    819232
True        528
Name: count, dtype: int64

In [829]:
#df_listings_clean.to_csv("../data/Listing/listings_cleaned.csv", index=False)